# Exercise 1: Financial PhraseBank

The goal here is to get familiar with a standard finance sentiment dataset and think about what would be a fair benchmark. The modeling choices are intentionally left open.


## What is in the data?

Each row is a short financial sentence labeled as:

- `0`: negative
- `1`: neutral
- `2`: positive

A useful reminder: in this dataset, sentiment is from an **investor / market interpretation**, not from a generic emotional tone.


In [ ]:
from pathlib import Path
import pandas as pd

candidates = [Path.cwd(), Path.cwd() / 'sessions' / 'in_class_assignment']
assignment_root = next(path for path in candidates if (path / 'data').exists())
data_dir = assignment_root / 'data' / 'financial_phrasebank'

label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

train_df = pd.read_parquet(data_dir / 'train.parquet')
validation_df = pd.read_parquet(data_dir / 'validation.parquet')
test_df = pd.read_parquet(data_dir / 'test.parquet')

for frame in (train_df, validation_df, test_df):
    frame['label_name'] = frame['label'].map(label_map)

print('train:', train_df.shape, '| validation:', validation_df.shape, '| test:', test_df.shape)


In [ ]:
class_balance = pd.concat(
    {
        'train': train_df['label_name'].value_counts().sort_index(),
        'validation': validation_df['label_name'].value_counts().sort_index(),
        'test': test_df['label_name'].value_counts().sort_index(),
    },
    axis=1,
)
class_balance


In [ ]:
examples = (
    train_df.groupby('label_name', group_keys=False)
    .head(2)
    [['label_name', 'sentence']]
    .reset_index(drop=True)
)
print(examples.to_string(index=False))


## Optional: a tiny `ZeroShotNLI` demo on 10 rows

This is **not** the benchmark itself. It is only a minimal illustration of how `FewShotX.ZeroShotNLI` can be used.
Leave `RUN_ZERO_SHOT_DEMO = False` unless the full transformer stack is available and you want to download the model.


In [ ]:
RUN_ZERO_SHOT_DEMO = False

if RUN_ZERO_SHOT_DEMO:
    import sys

    repo_root = assignment_root.parents[1]
    src_root = repo_root / 'src'
    if str(src_root) not in sys.path:
        sys.path.append(str(src_root))

    from FewShotX import ZeroShotNLI

    demo_df = test_df.head(10).copy()
    zsnli = ZeroShotNLI(batch_size=4, verbose=True)
    demo_scored = zsnli.score_df(
        demo_df,
        text_col='sentence',
        labels=['negative for investors', 'neutral for investors', 'positive for investors'],
        label_names=['score_negative', 'score_neutral', 'score_positive'],
    )
    score_cols = ['score_negative', 'score_neutral', 'score_positive']
    demo_scored['predicted_label'] = demo_scored[score_cols].idxmax(axis=1).str.replace('score_', '', regex=False)
    print(demo_scored[['sentence', 'label_name', 'predicted_label']].to_string(index=False))
else:
    print('ZeroShotNLI demo skipped. Set RUN_ZERO_SHOT_DEMO = True if you want to run it.')


## Suggested benchmark

Students can choose how much they want to implement, but a natural comparison is:

- `ZeroShotLearner` from `FewShotX`
- `ZeroShotNLI` from `FewShotX`
- `SetFit` as the few-shot benchmark

Good discussion points:

- How informative should the label prompts be?
- How many labeled examples per class are enough to start improving performance?
- At what point do gains become small relative to the extra labeling effort?
- Which mistakes remain hard even with labels?


## Ideas to leave open for students

1. Try a very small support set first, for example 4 or 8 examples per class.
2. Compare random support examples versus carefully selected informative ones.
3. Report at least `accuracy` and `macro F1`.
4. Keep a short note on which label prompts worked better and why.
